# Stage 1 Model Surgery for LLaMA vocab alignment

This notebook adjusts a student checkpoint to match the tokenizer size for the LLaMA 3.2 3B tokenizer. It ensures the embedding matrices are resized once so future training runs do not need to resize vocabularies at runtime.

In [ ]:
# Parameters
# Update these paths for your environment before running.
STUDENT_IN = "/path/to/current/student/checkpoint"
STUDENT_OUT_DIR = "/path/to/output/checkpoint-128256"
# Optional: set to a gs:// URI to upload the result. Leave as None to skip.
STUDENT_OUT_GCS = None


In [ ]:
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


In [ ]:
TOKENIZER_ID = "meta-llama/Llama-3.2-3B"

# Load the tokenizer and make sure padding is defined.
tok = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True)
pad_was_added = False
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
    pad_was_added = True

target = len(tok)
print(f"Tokenizer {TOKENIZER_ID} loaded with {target} tokens.")


In [ ]:
student_path = Path(STUDENT_IN)
if not student_path.exists():
    raise FileNotFoundError(f"Student checkpoint not found at {student_path}")

print(f"Loading student checkpoint from {student_path}...")
model = AutoModelForCausalLM.from_pretrained(
    student_path,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

current_vocab = model.get_input_embeddings().weight.size(0)
if current_vocab != target:
    print(f"Resizing token embeddings from {current_vocab} to {target}.")
    model.resize_token_embeddings(target)
    if hasattr(model, "tie_weights"):
        try:
            model.tie_weights()
        except ValueError as err:
            print(f"Warning: tie_weights raised ValueError: {err}")
else:
    print("Model embeddings already match tokenizer length; no resize needed.")

model.config.vocab_size = target


In [ ]:
input_embeddings = model.get_input_embeddings().weight.shape
output_embeddings = None
if hasattr(model, "get_output_embeddings") and model.get_output_embeddings() is not None:
    output_embeddings = model.get_output_embeddings().weight.shape

print("Tokenizer length:", len(tok))
print("Tokenizer vocab_size attribute:", getattr(tok, "vocab_size", None))
print("Input embedding weight shape:", input_embeddings)
print("LM head weight shape:", output_embeddings)

assert len(tok) == input_embeddings[0], "Tokenizer length and model embedding size must match"


In [ ]:
output_path = Path(STUDENT_OUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)

print(f"Saving surgically aligned checkpoint to {output_path}")
model.save_pretrained(output_path)

if pad_was_added:
    tok.save_pretrained(output_path)
    print("Tokenizer saved with updated padding token.")
else:
    print("Tokenizer padding already defined; skipping tokenizer save.")


In [ ]:
if STUDENT_OUT_GCS:
    if not STUDENT_OUT_GCS.startswith("gs://"):
        raise ValueError("STUDENT_OUT_GCS must be a gs:// URI")
    print(f"Uploading {output_path} to {STUDENT_OUT_GCS}")
    !gsutil -m cp -r "${output_path}" "${STUDENT_OUT_GCS}"
else:
    print("STUDENT_OUT_GCS not set; skipping upload.")


> **Note:** The checkpoint saved at `STUDENT_OUT_DIR` is now the source of truth for vocabulary sizing. Future training runs should load this adjusted checkpoint to avoid runtime resizing.
